In [10]:
from pathlib import Path
from datetime import datetime
from collections import Counter
import re
import unicodedata

import pandas as pd
from dateutil.relativedelta import relativedelta

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer


# ============================================================
# 2. DOWNLOAD NLTK RESOURCES
# ============================================================
# NLTK needs small data files for stopwords, lemmatization,
# and VADER sentiment analysis.

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("vader_lexicon")


# ============================================================
# 3. SET FILE PATHS
# ============================================================
# Keep _chat.txt in the same folder as this script.
# If your file is somewhere else, replace "_chat.txt" with the full path.

CHAT_FILE = Path("_chat.txt")

# This folder will store all output CSV files.
OUTPUT_FOLDER = Path("whatsapp_analysis_output")
OUTPUT_FOLDER.mkdir(exist_ok=True)


# ============================================================
# 4. READ RAW WHATSAPP TEXT FILE
# ============================================================

raw_text = CHAT_FILE.read_text(encoding="utf-8", errors="replace")

print("Raw file loaded successfully.")
print("Number of characters in raw file:", len(raw_text))


# ============================================================
# 5. REMOVE INVISIBLE WHATSAPP CHARACTERS
# ============================================================
# WhatsApp exports often contain invisible Unicode marks.
# These can break parsing, so we remove them.

def remove_invisible_characters(text):
    """
    Removes invisible WhatsApp formatting characters.

    Input:
        text: original text string

    Output:
        cleaned text string
    """

    invisible_chars = [
        "\u200e",  # left-to-right mark
        "\u202a",  # left-to-right embedding
        "\u202c",  # pop directional formatting
        "\u202d",
        "\u202e",
        "\u2066",
        "\u2067",
        "\u2068",
        "\u2069",
        "\ufeff",
    ]

    for char in invisible_chars:
        text = text.replace(char, "")

    return text


raw_text = remove_invisible_characters(raw_text)


# ============================================================
# 6. PARSE WHATSAPP EXPORT INTO STRUCTURED ROWS
# ============================================================
# WhatsApp format usually looks like:
# [07.10.25, 1:44:51 PM] Sender Name: Message text
#
# Some messages continue onto the next line.
# This parser handles those multi-line messages.

message_start_pattern = re.compile(
    r"^\[(\d{1,2}\.\d{1,2}\.\d{2}),\s*"
    r"(\d{1,2}:\d{2}:\d{2})\s*"
    r"([AP]M)?\]\s*"
    r"(.*?):\s*"
    r"(.*)$"
)

records = []
current_record = None

for line in raw_text.splitlines():
    match = message_start_pattern.match(line)

    if match:
        # Save the previous message before starting a new one.
        if current_record is not None:
            records.append(current_record)

        date_text = match.group(1)
        time_text = match.group(2)
        am_pm = match.group(3)
        sender = match.group(4).strip()
        message = match.group(5).strip()

        # Convert date/time text into a real Python datetime object.
        if am_pm:
            message_datetime = datetime.strptime(
                f"{date_text} {time_text} {am_pm}",
                "%d.%m.%y %I:%M:%S %p"
            )
        else:
            message_datetime = datetime.strptime(
                f"{date_text} {time_text}",
                "%d.%m.%y %H:%M:%S"
            )

        current_record = {
            "datetime": message_datetime,
            "sender_original": sender,
            "message_original": message
        }

    else:
        # This line is a continuation of the previous WhatsApp message.
        if current_record is not None:
            current_record["message_original"] += "\n" + line.strip()

# Add the final message.
if current_record is not None:
    records.append(current_record)

df = pd.DataFrame(records)

print("Parsed WhatsApp rows:", len(df))


# ============================================================
# 7. ADD BASIC TIME COLUMNS
# ============================================================

df["date"] = df["datetime"].dt.date
df["month"] = df["datetime"].dt.to_period("M").astype(str)
df["weekday"] = df["datetime"].dt.day_name()
df["hour"] = df["datetime"].dt.hour


# ============================================================
# 8. ANONYMIZE SENDERS
# ============================================================
# This protects private names and phone numbers in assignment output.
# Example:
# Vikrant Singh Thakur -> Sender_001

unique_senders = sorted(df["sender_original"].dropna().unique())

sender_mapping = {
    sender: f"Sender_{number:03d}"
    for number, sender in enumerate(unique_senders, start=1)
}

df["sender"] = df["sender_original"].map(sender_mapping)

sender_mapping_df = pd.DataFrame(
    list(sender_mapping.items()),
    columns=["sender_original", "sender_anonymous"]
)

sender_mapping_df.to_csv(
    OUTPUT_FOLDER / "sender_mapping_private_do_not_submit.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. IDENTIFY SYSTEM / MEDIA / DELETED MESSAGES
# ============================================================
# These are not useful for NLP sentiment analysis.
#
# Examples:
# - joined using a group link
# - image omitted
# - sticker omitted
# - This message was deleted

system_or_media_pattern = re.compile(
    r"joined using a group link|"
    r"was added|"
    r"\bleft\b|"
    r"created this group|"
    r"changed the group name|"
    r"changed the group description|"
    r"this group has over|"
    r"messages and calls are end-to-end encrypted|"
    r"anyone in this group can invite|"
    r"image omitted|"
    r"video omitted|"
    r"audio omitted|"
    r"sticker omitted|"
    r"gif omitted|"
    r"this message was deleted",
    flags=re.IGNORECASE
)

df["is_system_or_media"] = df["message_original"].str.contains(
    system_or_media_pattern,
    na=False
)


# ============================================================
# 10. FILTER LAST 6 MONTHS
# ============================================================
# We calculate "last 6 months" from the latest message inside
# the chat file. This is safer when the export is old.

latest_message_date = df["datetime"].max()
six_month_cutoff = latest_message_date - relativedelta(months=6)

df_last_6_months = df[df["datetime"] >= six_month_cutoff].copy()

print("Latest message date:", latest_message_date)
print("Six-month cutoff date:", six_month_cutoff)
print("Rows after last 6 months filter:", len(df_last_6_months))


# ============================================================
# 11. KEEP ONLY REAL USER-WRITTEN MESSAGES FOR NLP
# ============================================================

nlp_df = df_last_6_months[
    df_last_6_months["is_system_or_media"] == False
].copy()

print("Rows after removing system/media/deleted messages:", len(nlp_df))


# ============================================================
# 12. EXTENDED TEXT CLEANING
# ============================================================

url_pattern = re.compile(
    r"https?://\S+|www\.\S+",
    flags=re.IGNORECASE
)

email_pattern = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

phone_pattern = re.compile(
    r"(?:\+?\d[\d\s\-\(\)\u00a0]{7,}\d)"
)


def clean_private_information(text):
    """
    Replaces URLs, emails, and phone numbers with safe placeholders.

    Example:
        "Call me at +49 123456789"
        becomes:
        "Call me at PHONE"
    """

    text = str(text)

    # Normalize Unicode text.
    # This converts special versions of characters into standard forms.
    text = unicodedata.normalize("NFKC", text)

    # Replace sensitive patterns.
    text = url_pattern.sub(" URL ", text)
    text = email_pattern.sub(" EMAIL ", text)
    text = phone_pattern.sub(" PHONE ", text)

    return text


def clean_for_sentiment(text):
    """
    Cleans text for sentiment analysis.

    Important:
    We do NOT remove words like 'not' or "don't" here.
    Why?
    Because "I don't like this" must not become "I like this".
    """

    text = clean_private_information(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_for_nlp_tokens(text):
    """
    Cleans text for token-based NLP.

    This version:
    - lowercases
    - removes URLs/phones/emails
    - removes punctuation/numbers/symbols
    - keeps only alphabetic words
    """

    text = clean_private_information(text)
    text = text.lower()
    text = text.replace("\n", " ")

    # Keep only English alphabet characters and apostrophes.
    text = re.sub(r"[^a-zA-Z\s']", " ", text)

    # Remove repeated spaces.
    text = re.sub(r"\s+", " ", text).strip()

    return text


nlp_df["message_for_sentiment"] = nlp_df["message_original"].apply(clean_for_sentiment)
nlp_df["message_clean_nlp"] = nlp_df["message_original"].apply(clean_for_nlp_tokens)


# ============================================================
# 13. SPLIT MESSAGES INTO SHORTER SENTENCES
# ============================================================

def split_into_short_sentences(text, max_words=18):
    """
    Splits a message into shorter sentence-like parts.

    Why max_words?
    WhatsApp messages can be long.
    Shorter rows are easier to analyze for sentiment and keywords.

    Input:
        text: message string
        max_words: maximum words allowed in one short sentence

    Output:
        list of short sentence strings
    """

    text = str(text)
    text = text.replace("\n", ". ")

    # First split on sentence punctuation.
    rough_sentences = re.split(r"(?<=[.!?])\s+", text)

    short_sentences = []

    for sentence in rough_sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        words = sentence.split()

        # If the sentence is short, keep it.
        if len(words) <= max_words:
            short_sentences.append(sentence)

        # If the sentence is long, cut it into smaller chunks.
        else:
            for start_index in range(0, len(words), max_words):
                chunk = words[start_index:start_index + max_words]
                short_sentences.append(" ".join(chunk))

    return short_sentences


nlp_df["short_sentences"] = nlp_df["message_for_sentiment"].apply(
    split_into_short_sentences
)

# explode() turns one row with a list of sentences
# into many sentence rows.
sentence_df = nlp_df.explode("short_sentences").copy()

sentence_df = sentence_df.rename(
    columns={"short_sentences": "sentence_original"}
)

sentence_df["sentence_for_sentiment"] = sentence_df["sentence_original"].apply(
    clean_for_sentiment
)

sentence_df["sentence_clean_nlp"] = sentence_df["sentence_original"].apply(
    clean_for_nlp_tokens
)


# ============================================================
# 14. TOKENIZATION, STOPWORD REMOVAL, STEMMING, LEMMATIZATION
# ============================================================

english_stopwords = set(stopwords.words("english"))

# Extra WhatsApp/group-chat words that are usually not useful.
custom_stopwords = {
    "dont",
    "don't",
    "and",
    "so",
    "ok",
    "okay",
    "hi",
    "hello",
    "hey",
    "guys",
    "please",
    "thanks",
    "thank",
    "im",
    "i'm",
    "u",
    "ur",
    "anyone",
    "everyone",
    "yeah",
    "yes",
    "no"
}

all_stopwords = english_stopwords.union(custom_stopwords)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def tokenize_text(text):
    """
    Splits cleaned text into word tokens.

    Example:
        "hello everyone today"
        becomes:
        ["hello", "everyone", "today"]
    """

    return re.findall(r"[a-zA-Z']+", str(text))


def remove_stopwords_from_tokens(tokens):
    """
    Removes common words that usually carry little meaning.

    Example:
        ["this", "is", "good"]
        becomes:
        ["good"]
    """

    useful_tokens = []

    for token in tokens:
        token = token.strip().lower()

        if len(token) <= 1:
            continue

        if token in all_stopwords:
            continue

        useful_tokens.append(token)

    return useful_tokens


def stem_tokens(tokens):
    """
    Stemming cuts words down to a rough root.

    Example:
        studying, studies, studied
        may become:
        studi
    """

    return [stemmer.stem(token) for token in tokens]


def lemmatize_tokens(tokens):
    """
    Lemmatization converts words to dictionary-like base forms.

    Example:
        studies -> study
        classes -> class
    """

    return [lemmatizer.lemmatize(token) for token in tokens]


sentence_df["tokens"] = sentence_df["sentence_clean_nlp"].apply(tokenize_text)

sentence_df["tokens_no_stopwords"] = sentence_df["tokens"].apply(
    remove_stopwords_from_tokens
)

sentence_df["stems"] = sentence_df["tokens_no_stopwords"].apply(stem_tokens)

sentence_df["lemmas"] = sentence_df["tokens_no_stopwords"].apply(lemmatize_tokens)

sentence_df["preprocessed_text"] = sentence_df["lemmas"].apply(
    lambda tokens: " ".join(tokens)
)


# ============================================================
# 15. SENTIMENT ANALYSIS
# ============================================================
# VADER is good for short social/chat-style text.
#
# It returns a compound score from -1 to +1:
# - close to -1 = negative
# - close to 0  = neutral
# - close to +1 = positive

sentiment_analyzer = SentimentIntensityAnalyzer()


def get_sentiment_score(text):
    """
    Returns VADER compound sentiment score.
    """

    scores = sentiment_analyzer.polarity_scores(str(text))
    return scores["compound"]


def get_sentiment_label(score):
    """
    Converts numeric sentiment score into a readable label.
    """

    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"


sentence_df["sentiment_score"] = sentence_df["sentence_for_sentiment"].apply(
    get_sentiment_score
)

sentence_df["sentiment_label"] = sentence_df["sentiment_score"].apply(
    get_sentiment_label
)


# ============================================================
# 16. SUMMARY TABLES
# ============================================================

summary_table = pd.DataFrame({
    "metric": [
        "Total parsed WhatsApp rows",
        "Date of first parsed message",
        "Date of latest parsed message",
        "Last 6 months cutoff date",
        "Rows after last 6 months filter",
        "Rows used for NLP after removing system/media/deleted messages",
        "Unique senders in full chat",
        "Unique active senders in NLP dataset",
        "Average sentiment score"
    ],
    "value": [
        len(df),
        df["datetime"].min(),
        df["datetime"].max(),
        six_month_cutoff,
        len(df_last_6_months),
        len(nlp_df),
        df["sender"].nunique(),
        nlp_df["sender"].nunique(),
        round(sentence_df["sentiment_score"].mean(), 4)
    ]
})

sentiment_summary = (
    sentence_df["sentiment_label"]
    .value_counts()
    .reset_index()
)

sentiment_summary.columns = ["sentiment_label", "count"]

messages_by_sender = (
    nlp_df.groupby("sender")
    .size()
    .reset_index(name="message_count")
    .sort_values("message_count", ascending=False)
)

messages_by_month = (
    nlp_df.groupby("month")
    .size()
    .reset_index(name="message_count")
)

messages_by_weekday = (
    nlp_df.groupby("weekday")
    .size()
    .reset_index(name="message_count")
    .sort_values("message_count", ascending=False)
)

messages_by_hour = (
    nlp_df.groupby("hour")
    .size()
    .reset_index(name="message_count")
    .sort_values("message_count", ascending=False)
)


# ============================================================
# 17. TOP WORDS AFTER CLEANING
# ============================================================

all_lemmas = []

for token_list in sentence_df["lemmas"]:
    all_lemmas.extend(token_list)

word_counts = Counter(all_lemmas)

top_words = pd.DataFrame(
    word_counts.most_common(30),
    columns=["word", "count"]
)


# ============================================================
# 18. SIMPLE TOPIC KEYWORD COUNTS
# ============================================================

topic_keywords = {
    "Course/Admin/Deadlines": r"\b(deadline|report|assignment|exam|semester|class|lecture|schedule|prof|university|document)\b",
    "Social/Events/Greetings": r"\b(christmas|new year|merry|happy|party|event|birthday|morning|gather|join)\b",
    "Help/Questions/Logistics": r"\b(anyone|where|how|link|find|lost|cable|jacket|registration|office|room|arc|help)\b",
    "Internship/Jobs": r"\b(internship|job|career|linkedin|resume|cv)\b",
    "Technical/AI/Data": r"\b(ai|data|engineering|science|analysis|process|python|machine|learning)\b",
}

topic_rows = []

for topic_name, keyword_pattern in topic_keywords.items():
    count = sentence_df["sentence_clean_nlp"].str.contains(
        keyword_pattern,
        regex=True,
        case=False,
        na=False
    ).sum()

    topic_rows.append({
        "topic": topic_name,
        "matching_sentence_count": count
    })

topic_summary = pd.DataFrame(topic_rows).sort_values(
    "matching_sentence_count",
    ascending=False
)


# ============================================================
# 19. SAVE OUTPUT FILES
# ============================================================

df.to_csv(
    OUTPUT_FOLDER / "01_full_parsed_chat.csv",
    index=False,
    encoding="utf-8-sig"
)

df_last_6_months.to_csv(
    OUTPUT_FOLDER / "02_last_6_months_all_rows.csv",
    index=False,
    encoding="utf-8-sig"
)

nlp_df.to_csv(
    OUTPUT_FOLDER / "03_last_6_months_clean_messages.csv",
    index=False,
    encoding="utf-8-sig"
)

sentence_df.to_csv(
    OUTPUT_FOLDER / "04_sentence_level_preprocessed_sentiment.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_table.to_csv(
    OUTPUT_FOLDER / "05_summary_table.csv",
    index=False,
    encoding="utf-8-sig"
)

sentiment_summary.to_csv(
    OUTPUT_FOLDER / "06_sentiment_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

messages_by_sender.to_csv(
    OUTPUT_FOLDER / "07_messages_by_sender.csv",
    index=False,
    encoding="utf-8-sig"
)

messages_by_month.to_csv(
    OUTPUT_FOLDER / "08_messages_by_month.csv",
    index=False,
    encoding="utf-8-sig"
)

messages_by_weekday.to_csv(
    OUTPUT_FOLDER / "09_messages_by_weekday.csv",
    index=False,
    encoding="utf-8-sig"
)

messages_by_hour.to_csv(
    OUTPUT_FOLDER / "10_messages_by_hour.csv",
    index=False,
    encoding="utf-8-sig"
)

top_words.to_csv(
    OUTPUT_FOLDER / "11_top_words.csv",
    index=False,
    encoding="utf-8-sig"
)

topic_summary.to_csv(
    OUTPUT_FOLDER / "12_topic_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 20. PRINT QUICK RESULTS
# ============================================================

print("\n==============================")
print("ANALYSIS COMPLETE")
print("==============================")

print("\nSummary:")
print(summary_table)

print("\nSentiment summary:")
print(sentiment_summary)

print("\nTop 10 most active anonymous senders:")
print(messages_by_sender.head(10))

print("\nTop 20 words after preprocessing:")
print(top_words.head(20))

print("\nTopic summary:")
print(topic_summary)

print("\nOutput folder:")
print(OUTPUT_FOLDER.resolve())

[nltk_data] Downloading package stopwords to C:\Users\Vikrant Singh
[nltk_data]     Thakur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Vikrant Singh
[nltk_data]     Thakur\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Vikrant Singh
[nltk_data]     Thakur\AppData\Roaming\nltk_data...
[nltk_data] Downloading package vader_lexicon to C:\Users\Vikrant
[nltk_data]     Singh Thakur\AppData\Roaming\nltk_data...


FileNotFoundError: [Errno 2] No such file or directory: '_chat.txt'